In [3]:
import mdtraj as md
import os
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
from deeptime.decomposition import TICA

TICA - time-lagged independent component analysis. Map data to the "slow" processes. Find coordinates that are most correlated with themselves (autocorrelation) after a given time lag, tau. Tends to filter out fast thermal noise and isolate slow, rare conformational changes.

In [2]:
# load trajectories and featurize
# using sparse, dry trajectories

top = "/n/home00/ewang03/scratch/GroEL/runs/2026-11-07_58169745/m20i_nadph_tmp_dry.pdb"
traj_files = ["/n/home00/ewang03/lab/GroEL/results/m20i_1us_2026-11-07/m20i_sparse_dry_01.xtc",
         "/n/home00/ewang03/lab/GroEL/results/m20i_1us_2026-11-07/m20i_sparse_dry_02.xtc",
         "/n/home00/ewang03/lab/GroEL/results/m20i_1us_2026-11-07/m20i_sparse_dry_03.xtc",
         "/n/home00/ewang03/lab/GroEL/results/m20i_1us_2026-11-07/m20i_sparse_dry_05.xtc"]

trajs = [md.load(f, top=top) for f in traj_files]

_, pairs = md.compute_contacts(trajs[0][0], scheme='ca')

def featurize(traj):
    dist, _ = md.compute_contacts(traj, contacts=pairs, scheme='ca')
    return dist

features = [featurize(t) for t in trajs]

In [ ]:
# pca to shrink the features. ~12,000 pairs and 4000 frames --> the matrix is severely underdetermined

X = np.concatenate(features)
pca_pre = PCA(n_components=100).fit(X) # shrink to 100 features (PCs) that are linear combinations of the original features
features_red = [pca_pre.transform(f) for f in features]
#print(features_red[0].shape)

(1000, 100)


In [8]:
dt_ns = 1 # time between frames
lag = 20 # lag time in frames (20 ns)

tica = TICA(lagtime=lag, var_cutoff=0.95, scaling='kinetic_map') # build TICA, want to cover 95% of kinetic variance. 
                                                                # scale each component by its eigenvalue
tica_model = tica.fit(features_red).fetch_model() # passing a list keeps runs separate 

tica_out = [tica_model.transform(x) for x in features_red] # return list of (n_frames, n_tics), one for each of the four trajectories
print(tica_out[0].shape)

# Implied timescales of the slow processes: t_i = -lag / ln(lambda_i)
its = tica_model.timescales() * dt_ns
print("Slowest timescales (ns):", its[:5])

(1000, 46)
Slowest timescales (ns): [315.82540601 126.20458465  75.72211078  54.17606179  35.4297895 ]
